# 07. 앙상블 모델과 성능 평가
RandomForest, ExtraTrees, GradientBoosting, HistGradientBoosting, XGBoost를 비교하고 GridSearchCV 튜닝과 분류 평가를 한 흐름으로 정리합니다.

> 데이터 준비는 한 셀에 묶고, 이 노트에서는 **앙상블 방식의 차이·모델 비교·튜닝·평가**에 집중합니다.

## 배깅과 부스팅
- **배깅(Bagging)**: 여러 모델을 독립적으로 학습하고 결과를 결합합니다. RandomForest, ExtraTrees가 대표적입니다.
- **부스팅(Boosting)**: 앞선 모델의 오류를 다음 모델이 보완하도록 순차적으로 학습합니다. GradientBoosting, HistGradientBoosting, XGBoost가 대표적입니다.

앙상블 모델이 항상 단일 모델보다 좋은 것은 아니며, 데이터와 튜닝 조건에 따라 비교가 필요합니다.

In [1]:
from common.libraries import *

# 이번 학습에서 집중할 기능
# - RandomForest / ExtraTrees : 배깅 계열 앙상블
# - GradientBoosting / HistGradientBoosting / XGBoost : 부스팅 계열 앙상블
# - GridSearchCV : 모델별 하이퍼파라미터 탐색
# - accuracy / precision / recall / F1 : 검증 데이터 기준 성능 비교

In [ ]:
# 기본 준비: 정제 → X/y → 6:2:2 분할 → 표준화
wine = pd.read_csv("./data/08_wine_exe.csv")

wine = wine.dropna().reset_index(drop=True)
wine = wine[wine["alcohol"] < 100].reset_index(drop=True)
wine = wine[wine["sugar"] < 100].reset_index(drop=True)

X = wine[["alcohol", "sugar", "pH"]]
y = wine["class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train)
val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(X_test)

print("Train / Val / Test :", X_train.shape, X_val.shape, X_test.shape)

## 기본 모델 비교
아직 튜닝하지 않은 기본 모델을 같은 데이터로 비교합니다.  
검증 정확도를 우선해서 보고, 훈련-검증 정확도 차이는 과적합을 확인하는 보조 지표로 사용합니다.

In [ ]:
models = [
    RandomForestClassifier(random_state=42, n_jobs=-1),
    ExtraTreesClassifier(random_state=42, n_jobs=-1),
    GradientBoostingClassifier(random_state=42),
    HistGradientBoostingClassifier(random_state=42),
    XGBClassifier(random_state=42, n_jobs=-1),
]

baseline = []

for model in models:
    model_name = model.__class__.__name__
    model.fit(train_scaled, y_train)

    train_pred = model.predict(train_scaled)
    val_pred = model.predict(val_scaled)
    test_pred = model.predict(test_scaled)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)

    baseline.append([
        model_name,
        "default",
        train_acc,
        val_acc,
        test_acc,
        train_acc - val_acc,
        precision_score(y_val, val_pred),
        recall_score(y_val, val_pred),
        f1_score(y_val, val_pred),
    ])

baseline_df = pd.DataFrame(
    baseline,
    columns=[
        "모델명", "경우의수",
        "훈련정확도", "검증정확도", "테스트정확도",
        "훈련-검증정확도차이",
        "검증정밀도", "검증재현율", "검증F1",
    ]
)

baseline_df.sort_values(
    ["검증정확도", "훈련-검증정확도차이"],
    ascending=[False, True],
    key=lambda s: s.abs() if s.name == "훈련-검증정확도차이" else s
)

## GridSearchCV 튜닝
모델마다 지원하는 하이퍼파라미터가 다르므로 별도로 정의합니다.

> `HistGradientBoostingClassifier`는 `n_estimators` 대신 `max_iter`를 사용합니다.  
> XGBoost의 `max_depth`는 정수값으로 지정합니다.

In [ ]:
param_grids = {
    "RandomForestClassifier": {
        "n_estimators": [50, 100, 150],
        "max_depth": [None, 10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2, 4],
    },
    "ExtraTreesClassifier": {
        "n_estimators": [50, 100, 150],
        "max_depth": [None, 10],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2, 4],
    },
    "GradientBoostingClassifier": {
        "n_estimators": [50, 100, 150],
        "max_depth": [2, 3, 5],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2, 4],
    },
    "HistGradientBoostingClassifier": {
        "max_iter": [50, 100, 150],
        "max_depth": [None, 10],
        "min_samples_leaf": [5, 10, 20],
    },
    "XGBClassifier": {
        "n_estimators": [50, 100, 150],
        "max_depth": [3, 6, 10],
        "min_child_weight": [1, 2, 4],
    },
}

In [ ]:
tuned = []
best_estimators = {}

for model in models:
    model_name = model.__class__.__name__

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[model_name],
        scoring="accuracy",
        cv=5,
        n_jobs=-1,
    )
    grid.fit(train_scaled, y_train)

    best_model = grid.best_estimator_
    best_estimators[model_name] = best_model

    train_pred = best_model.predict(train_scaled)
    val_pred = best_model.predict(val_scaled)
    test_pred = best_model.predict(test_scaled)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)

    tuned.append([
        model_name,
        grid.best_params_,
        train_acc,
        val_acc,
        test_acc,
        train_acc - val_acc,
        precision_score(y_val, val_pred),
        recall_score(y_val, val_pred),
        f1_score(y_val, val_pred),
    ])

tuned_df = pd.DataFrame(
    tuned,
    columns=[
        "모델명", "경우의수",
        "훈련정확도", "검증정확도", "테스트정확도",
        "훈련-검증정확도차이",
        "검증정밀도", "검증재현율", "검증F1",
    ]
)

tuned_df.sort_values(
    ["검증정확도", "훈련-검증정확도차이"],
    ascending=[False, True],
    key=lambda s: s.abs() if s.name == "훈련-검증정확도차이" else s
)

## 최종 모델 확인
이 학습 노트에서는 **검증 정확도를 우선**하고, 값이 비슷하면 훈련-검증 차이와 F1 등을 함께 확인합니다.  
테스트 데이터는 최종 일반화 성능 확인용으로 사용합니다.

In [ ]:
final_df = tuned_df.copy()
final_df["과적합절대차이"] = final_df["훈련-검증정확도차이"].abs()

best_row = final_df.sort_values(
    ["검증정확도", "과적합절대차이", "검증F1"],
    ascending=[False, True, False]
).iloc[0]

print("최종 모델 :", best_row["모델명"])
print("최적 파라미터 :", best_row["경우의수"])
print("검증 정확도 :", best_row["검증정확도"])
print("검증 F1 :", best_row["검증F1"])
print("테스트 정확도 :", best_row["테스트정확도"])

## 오차행렬로 결과 확인

In [ ]:
best_model = best_estimators[best_row["모델명"]]
test_pred = best_model.predict(test_scaled)

cm = confusion_matrix(y_test, test_pred, labels=best_model.classes_)
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=best_model.classes_
).plot()

plt.show()